# TUGAS MANDIRI

### Konteks / Skenario

Platform "Kopi Nusantara" (skenario yang sama dari Tugas Mandiri Pertemuan 7) ingin mulai menyimpan katalog menu mereka di MongoDB — karena setiap cabang punya menu musiman/spesial dengan atribut yang berbeda-beda (tidak semua menu punya atribut yang sama, misalnya hanya menu kopi yang punya atribut "tingkat_kepahitan", sementara menu makanan punya atribut "pedas_level"). Kalian ditugaskan membangun dan menganalisis katalog ini.

In [1]:
from pymongo import MongoClient
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import avg

client = MongoClient("mongodb://localhost:27017/")
db = client["kopi_nusantara"]
koleksi_menu = db["menu"]

koleksi_menu.delete_many({})

DeleteResult({'n': 0, 'ok': 1.0}, acknowledged=True)

### A. Membangun Katalog (Create) 

Buat database baru bernama kopi_nusantara, collection bernama menu. Sisipkan minimal 12 dokumen menu dengan field bervariasi sesuai kategorinya:

   1.  Menu kategori "Kopi": wajib memiliki field nama_menu, harga, kategori, dan tingkat_kepahitan (skala 1-5).
   2.  Menu kategori "Makanan": wajib memiliki field nama_menu, harga, kategori, dan pedas_level (skala 0-3).
   3.  Boleh menambahkan field lain sesuai kreativitas anda (mis. stok, rekomendasi_barista).

In [2]:
data_menu = [
    # Kategori Kopi (wajib ada tingkat_kepahitan)
    {"nama_menu": "Espresso Single", "kategori": "Kopi", "harga": 18000, "tingkat_kepahitan": 5, "stok": 50},
    {"nama_menu": "Americano", "kategori": "Kopi", "harga": 22000, "tingkat_kepahitan": 4, "stok": 40},
    {"nama_menu": "Kopi Gula Aren", "kategori": "Kopi", "harga": 25000, "tingkat_kepahitan": 2, "stok": 100},
    {"nama_menu": "Latte Macchiato", "kategori": "Kopi", "harga": 28000, "tingkat_kepahitan": 2, "stok": 35},
    {"nama_menu": "Cold Brew Dark", "kategori": "Kopi", "harga": 32000, "tingkat_kepahitan": 5, "stok": 20},
    
    # Kategori Makanan (wajib ada pedas_level)
    {"nama_menu": "Nasi Goreng Rempah", "kategori": "Makanan", "harga": 28000, "pedas_level": 2},
    {"nama_menu": "Mie Goreng Jawa", "kategori": "Makanan", "harga": 24000, "pedas_level": 1},
    {"nama_menu": "Ayam Geprek Sambal Ijo", "kategori": "Makanan", "harga": 30000, "pedas_level": 3},
    {"nama_menu": "Cireng Bumbu Rujak", "kategori": "Makanan", "harga": 14000, "pedas_level": 1},
    {"nama_menu": "Toast Cokelat Keju", "kategori": "Makanan", "harga": 18000, "pedas_level": 0},

    # Kategori Non-Kopi / Minuman Lain
    {"nama_menu": "Matcha Latte", "kategori": "Non-Kopi", "harga": 27000, "rekomendasi_barista": True},
    {"nama_menu": "Earl Grey Tea", "kategori": "Non-Kopi", "harga": 16000, "rekomendasi_barista": False}
]

koleksi_menu.insert_many(data_menu)
print("Berhasil menyisipkan data menu ke MongoDB!\n")

Berhasil menyisipkan data menu ke MongoDB!



### B. Kueri Bertingkat (Read) 

Tampilkan hasil dari tiga kueri berikut :

   1. Seluruh menu kategori "Kopi" dengan tingkat_kepahitan di atas 3.
   2. Seluruh menu dengan harga di antara 15000 dan 30000.
   3. Menu yang tidak memiliki field pedas_level sama sekali.

In [3]:
print("--- 1. Menu Kopi dengan tingkat kepahitan > 3 ---")
for m in koleksi_menu.find({"kategori": "Kopi", "tingkat_kepahitan": {"$gt": 3}}):
    print(f"- {m['nama_menu']} (Kepahitan: {m['tingkat_kepahitan']})")

print("\n--- 2. Menu dengan harga antara 15.000 dan 30.000 ---")
for m in koleksi_menu.find({"harga": {"$gte": 15000, "$lte": 30000}}):
    print(f"- {m['nama_menu']} (Rp{m['harga']:,})")

print("\n--- 3. Menu yang TIDAK memiliki field pedas_level ---")
for m in koleksi_menu.find({"pedas_level": {"$exists": False}}):
    print(f"- {m['nama_menu']} [{m['kategori']}]")

--- 1. Menu Kopi dengan tingkat kepahitan > 3 ---
- Espresso Single (Kepahitan: 5)
- Americano (Kepahitan: 4)
- Cold Brew Dark (Kepahitan: 5)

--- 2. Menu dengan harga antara 15.000 dan 30.000 ---
- Espresso Single (Rp18,000)
- Americano (Rp22,000)
- Kopi Gula Aren (Rp25,000)
- Latte Macchiato (Rp28,000)
- Nasi Goreng Rempah (Rp28,000)
- Mie Goreng Jawa (Rp24,000)
- Ayam Geprek Sambal Ijo (Rp30,000)
- Toast Cokelat Keju (Rp18,000)
- Matcha Latte (Rp27,000)
- Earl Grey Tea (Rp16,000)

--- 3. Menu yang TIDAK memiliki field pedas_level ---
- Espresso Single [Kopi]
- Americano [Kopi]
- Kopi Gula Aren [Kopi]
- Latte Macchiato [Kopi]
- Cold Brew Dark [Kopi]
- Matcha Latte [Non-Kopi]
- Earl Grey Tea [Non-Kopi]


### C. Pembaruan Data (Update) 

Tambahkan field status bernilai "Tersedia" ke seluruh dokumen menggunakan update_many(). Kemudian, ubah status menjadi "Habis" khusus untuk 2 menu pilihan anda menggunakan update_one().

In [4]:
# 1. Tambah field status = "Tersedia" ke seluruh dokumen
koleksi_menu.update_many({}, {"$set": {"status": "Tersedia"}})

# 2. Ubah status jadi "Habis" untuk 2 menu pilihan
koleksi_menu.update_one({"nama_menu": "Cold Brew Dark"}, {"$set": {"status": "Habis"}})
koleksi_menu.update_one({"nama_menu": "Cireng Bumbu Rujak"}, {"$set": {"status": "Habis"}})

print("\nStatus berhasil diperbarui!")


Status berhasil diperbarui!


### D. Integrasi ke Spark 

Ambil seluruh data dari collection menu, ubah menjadi Spark DataFrame (ikuti pola Sub-bab 9.6 — perhatikan field yang mungkin kosong/null pada sebagian dokumen, seperti tingkat_kepahitan yang hanya dimiliki menu Kopi). Tampilkan:

   1. printSchema() hasil konversi.
   2. Rata-rata harga per kategori menggunakan groupBy.

In [8]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import avg
import pandas as pd

# Inisialisasi SparkSession
spark = SparkSession.builder \
    .appName("Tugas9-KopiNusantara") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# Langkah 1: Ambil seluruh dokumen dari MongoDB sebagai list of dict
dokumen_mongo = list(koleksi_menu.find({}))

# Langkah 2: Ubah menjadi pandas DataFrame
df_pandas = pd.DataFrame(dokumen_mongo)

# Langkah 3: Tipe data khusus MongoDB (_id, dan penanganan nilai kosong/NaN)
df_pandas["_id"] = df_pandas["_id"].astype(str)

# PENANGANAN ERROR: Isi nilai NaN/None agar PySpark tidak gagal inferensi schema
df_pandas = df_pandas.fillna({
    "tingkat_kepahitan": 0,
    "pedas_level": 0,
    "rekomendasi_barista": False,
    "stok": 0
})

# Langkah 4: Konversi ke Spark DataFrame
df_spark_menu = spark.createDataFrame(df_pandas)

# 1. Tampilkan printSchema()
print("--- 1. printSchema() Hasil Konversi ---")
df_spark_menu.printSchema()

# 2. Tampilkan Rata-rata Harga per Kategori
print("\n--- 2. Rata-rata Harga per Kategori ---")
df_spark_menu.groupBy("kategori").agg(
    avg("harga").alias("rata_rata_harga")
).show()

# Tutup SparkSession
spark.stop()
print("SparkSession ditutup.")

/tmp/ipykernel_11205/646112119.py:25: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_pandas = df_pandas.fillna({


--- 1. printSchema() Hasil Konversi ---
root
 |-- _id: string (nullable = true)
 |-- nama_menu: string (nullable = true)
 |-- kategori: string (nullable = true)
 |-- harga: long (nullable = true)
 |-- tingkat_kepahitan: double (nullable = true)
 |-- stok: double (nullable = true)
 |-- status: string (nullable = true)
 |-- pedas_level: double (nullable = true)
 |-- rekomendasi_barista: boolean (nullable = true)


--- 2. Rata-rata Harga per Kategori ---


+--------+---------------+
|kategori|rata_rata_harga|
+--------+---------------+
|    Kopi|        25000.0|
| Makanan|        22800.0|
|Non-Kopi|        21500.0|
+--------+---------------+

SparkSession ditutup.


### E. Kesimpulan 

Tulis pada markdown cell (minimal 100 kata): apa tantangan yang anda temui saat mengonversi data MongoDB (dengan field yang tidak seragam antar dokumen) menjadi Spark DataFrame (yang mengharuskan skema tabel seragam)? Bagaimana anda mengatasinya?

Tantangan utama saat mengonversi data dari MongoDB ke Spark DataFrame terletak pada ketidakseragaman atribut (heterogeneous schema) antar dokumen. Pada MongoDB, dokumen kategori "Kopi" memiliki atribut tingkat_kepahitan, sementara kategori "Makanan" memiliki atribut pedas_level. Karena Spark DataFrame mewajibkan struktur tabel dengan skema yang seragam pada setiap barisnya, dokumen yang tidak memiliki atribut tertentu akan dikonversi menjadi nilai null atau NaN oleh Spark.

Untuk mengatasi hal tersebut, langkah penanganan dilakukan pada tahap perantara menggunakan Pandas. Kita memastikan seluruh dokumen yang diambil dari PyMongo dimuat terlebih dahulu ke dalam Pandas DataFrame. Pandas secara otomatis menyelaraskan semua field yang ada di seluruh dokumen dan mengisi nilai yang absen dengan tipe data NaN/None. Setelah skema tabel perantara ini konsisten, proses konversi ke Spark DataFrame dapat berjalan dengan mulus tanpa mengalami kegagalan tipe data (schema mismatch).